# Agente de revisión de documentos y planos de ingeniería (modelo gratis en Kaggle)
Corre **sin API ni costo**: un modelo abierto con visión (Qwen2.5-VL) dentro de la GPU de Kaggle. Revisa documentos (PDF con texto, DOCX, XLSX, CSV, TXT, MD) **y planos** (PDF impresos desde DWG, PNG/JPG), y luego puedes hacerle preguntas.

Antes de ejecutar:
1. *Settings → Accelerator*: **GPU T4 x2** (o P100).
2. *Settings → Internet*: **On** (solo para descargar el modelo y los paquetes la primera vez).
3. Sube tus archivos como Dataset.
4. (Opcional) Sube `doc_review_agent.py`, `plan_review.py`, `local_llm.py`, `doc_chat.py`, `checklist_default.md` y `checklist_planos.md` como Dataset (o al panel de archivos del notebook).
Los planos se detectan solos (PDF de formato A3 o mayor, o con «plano/lámina/dwg» en el nombre). Si tus planos son A4, usa `ReviewConfig(plan_mode="always")`.

In [ ]:
# Los avisos rojos 'dependency resolver ... google-colab ...' son normales en Kaggle y no impiden continuar.
!pip install -q "transformers>=4.49" accelerate bitsandbytes pypdf python-docx openpyxl pymupdf pillow

In [ ]:
# Código del agente: se descarga/actualiza desde GitHub (siempre la última versión).
# Solo si no hay Internet se usan los .py que estén en /kaggle/input o /kaggle/working.
import glob, os, subprocess, sys
CODE_DIR = "/kaggle/working/agente"
if os.path.isdir(os.path.join(CODE_DIR, ".git")):
    r = subprocess.run(["git", "-C", CODE_DIR, "pull", "--ff-only", "-q"], capture_output=True, text=True)
else:
    r = subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/gallant-fermat-x83ohv",
                        "https://github.com/mrubilar84-ship-it/PRUEBA-AGENTE_1", CODE_DIR], capture_output=True, text=True)
if r.returncode:
    hits = glob.glob("/kaggle/input/**/doc_review_agent.py", recursive=True)
    if not hits:
        raise RuntimeError("No pude descargar el código (¿Internet: On?) ni hay copia en /kaggle/input:\n" + r.stderr)
    CODE_DIR = os.path.dirname(hits[0])
    print("Sin conexión a GitHub: uso la copia de", CODE_DIR)
sys.path.insert(0, CODE_DIR)
print("Código en:", CODE_DIR)

from doc_review_agent import ReviewConfig, review_folder
from local_llm import LocalLLM, LocalVLM

In [ ]:
# ¿Tienes PLANOS (dibujos) que revisar con este modelo? Si solo son documentos de texto / PDF con texto, deja False:
# es más rápido y liviano. Con True se carga un modelo que además ve imágenes.
CON_PLANOS = False

import torch
assert torch.cuda.is_available(), "Activa la GPU: panel derecho → Session options → Accelerator → GPU T4 x2"

if CON_PLANOS:
    llm = LocalVLM("Qwen/Qwen2.5-VL-7B-Instruct", load_4bit=True)   # ve imágenes (~7 GB en 4 bits)
else:
    llm = LocalLLM("Qwen/Qwen2.5-7B-Instruct", load_4bit=True)      # solo texto (~5 GB en 4 bits)
# Más rápido y liviano: "Qwen/Qwen2.5-3B-Instruct". Más preciso y más lento: "Qwen/Qwen2.5-14B-Instruct".

print(llm.generate_text("Responde en una palabra.", "Di: listo"))   # prueba rápida: debe imprimir "listo"

In [ ]:
cfg = ReviewConfig(
    plan_tiles=True,   # False = más rápido (solo vista general + cajetín)
    extra_instructions="Proyecto: <nombre>. Normas aplicables: <ej. AISC 360-16, NCh433>. Unidades: SI.",
)
IN_DIR = "/kaggle/input/documentos-ingenieria"   # nombre o URL de tu dataset (o una carpeta de /kaggle/input)
resultado = review_folder(IN_DIR, "/kaggle/working/informe", cfg, llm=llm)

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("/kaggle/working/informe/informe.md", encoding="utf-8").read()))

## Chat con los documentos
Haz preguntas, pide resúmenes o consulta los hallazgos de la revisión. Responde solo con lo que encuentra en los documentos y cita archivo y página.

## Interpretación de planos
Extrae de cada hoja qué muestra: tipo de plano, cajetín, elementos, cotas, materiales, notas y referencias. Se guarda en `planos_interpretados.md/json` y alimenta el chat (si tu carpeta no tiene planos, omite esta celda y la línea `add_plans`).

In [ ]:
from plan_review import interpret_folder
planos = interpret_folder(IN_DIR, "/kaggle/working/informe", cfg, llm=llm)
from IPython.display import Markdown, display
display(Markdown(open("/kaggle/working/informe/planos_interpretados.md", encoding="utf-8").read()))

In [ ]:
from doc_chat import DocChat
chat = DocChat(llm)
chat.add_folder(IN_DIR)
chat.add_review("/kaggle/working/informe/revision.json")   # opcional: permite preguntar por los hallazgos
chat.add_plans(planos)   # planos interpretados: el chat los usa y mira la hoja relevante

In [ ]:
print(chat.ask("¿Cuáles son los hallazgos críticos y mayores?"))
print(chat.ask("¿Qué norma se aplica y qué factores de seguridad se usan?"))
print(chat.ask("¿Qué muestra el plano de la hoja 1 y cuáles son sus cotas principales?"))
print(chat.ask("¿Qué materiales y normas se indican en los planos?"))

In [ ]:
print(chat.summarize())            # todos los documentos; o chat.summarize("archivo.pdf")

In [ ]:
# Preguntas sobre lo que SE VE en una hoja de plano (PDF y número de hoja)
print(chat.ask_image("¿Las cotas parciales suman la cota total? ¿Qué datos faltan en el cajetín?",
                     "/kaggle/input/documentos-ingenieria/plano.pdf", page=1))

In [ ]:
chat.repl()   # chat interactivo: escribe tus preguntas; /resumen, /reset, /salir